# Clase 3 · Laboratorio 2
## MLflow en Databricks: Tracking de experimentos y registro de modelos

**DevSeniorCode — Máster en Inteligencia Artificial & Data Science**
Módulo 3 · Unidad 3 · Clase 3

---

### Objetivo del laboratorio
1. Configurar un **Experiment** de MLflow.
2. Entrenar varias variantes de un modelo y registrar **parámetros, métricas y artefactos** con `mlflow.start_run()`.
3. Comparar ejecuciones (*runs*) y elegir el mejor modelo.
4. Registrar el modelo ganador en el **Model Registry** y moverlo a `Staging`.
5. Cargar el modelo registrado y usarlo para hacer predicciones — tal como lo haría un servicio en producción.

> **Cómo usar este notebook:** impórtalo a tu Workspace de **Databricks Free Edition** y conéctalo a **Serverless** — `mlflow` ya viene preinstalado, sin necesidad de configurar un cluster. Ejecuta las celdas en orden.


In [ ]:
# En Databricks Runtime ML, mlflow y scikit-learn ya vienen preinstalados.
# Si corres este notebook fuera de Databricks, descomenta la siguiente línea:
# %pip install mlflow scikit-learn pandas

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

print("Versión de MLflow:", mlflow.__version__)


## 1. Preparando los datos

Usamos el dataset **Wine** de scikit-learn (clasificación de tipos de vino según sus propiedades químicas) — pequeño, ideal para demostrar el flujo completo de MLflow sin depender de descargas externas.

In [ ]:
data = load_wine(as_frame=True)
X = data.data
y = data.target

print("Filas:", X.shape[0], "| Columnas:", X.shape[1])
print("Clases:", list(data.target_names))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape[0], "| Test:", X_test.shape[0])
X_train.head()


## 2. Creando (o seleccionando) el Experiment

En Databricks, cada notebook puede asociarse a un **Experiment** de MLflow. Si el experimento no existe, se crea automáticamente. En Databricks Free Edition, la ruta debe estar dentro de una carpeta accesible del Workspace (por ejemplo `/Shared/...` o tu carpeta de usuario).

In [ ]:
EXPERIMENT_NAME = "/Shared/clase3_mlflow_wine_demo"

try:
    mlflow.set_experiment(EXPERIMENT_NAME)
    print(f"Experiment activo: {EXPERIMENT_NAME}")
except Exception as e:
    # Fuera de Databricks, MLflow usa una carpeta local ./mlruns
    mlflow.set_experiment("clase3_mlflow_wine_demo_local")
    print("Ejecutando con tracking local (./mlruns) — fuera de un Workspace de Databricks.")


## 3. Tracking manual: `log_param`, `log_metric`, `log_model`

Entrenamos **tres variantes** de un `RandomForestClassifier`, cambiando sus hiperparámetros, y registramos cada intento como un *run* independiente dentro del mismo experimento.

In [ ]:
def entrenar_y_registrar(n_estimators, max_depth, run_name):
    with mlflow.start_run(run_name=run_name) as run:
        # 1) Parámetros del modelo
        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)
        mlflow.log_param("modelo", "RandomForestClassifier")

        # 2) Entrenamiento
        model = RandomForestClassifier(
            n_estimators=n_estimators, max_depth=max_depth, random_state=42
        )
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        # 3) Métricas de desempeño
        acc = accuracy_score(y_test, preds)
        f1 = f1_score(y_test, preds, average="macro")
        precision = precision_score(y_test, preds, average="macro")
        recall = recall_score(y_test, preds, average="macro")

        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)

        # 4) Artefacto: el modelo entrenado, versionado por MLflow
        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"[{run_name}] run_id={run.info.run_id} | accuracy={acc:.4f} | f1={f1:.4f}")
        return run.info.run_id, acc, f1


run_ids = []
run_ids.append(entrenar_y_registrar(n_estimators=50,  max_depth=3,  run_name="rf_pequeno"))
run_ids.append(entrenar_y_registrar(n_estimators=150, max_depth=6,  run_name="rf_mediano"))
run_ids.append(entrenar_y_registrar(n_estimators=300, max_depth=None, run_name="rf_grande"))


## 4. Autologging: la forma rápida

`mlflow.autolog()` registra automáticamente parámetros, métricas y el modelo — sin necesidad de llamar `log_param`/`log_metric` uno por uno. Ideal para prototipar rápido; en clase ya vimos ambos enfoques.

In [ ]:
mlflow.sklearn.autolog(log_models=True)

with mlflow.start_run(run_name="rf_autolog_demo"):
    model_auto = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
    model_auto.fit(X_train, y_train)
    preds_auto = model_auto.predict(X_test)
    # Con autolog, MLflow ya calculó y registró las métricas estándar por nosotros.
    print("Accuracy (calculada manualmente para verificar):", accuracy_score(y_test, preds_auto))

mlflow.sklearn.autolog(disable=True)  # lo desactivamos para volver al modo manual


## 5. Comparando ejecuciones (*runs*)

`mlflow.search_runs()` trae todas las ejecuciones del experimento activo como un DataFrame de Pandas — perfecto para comparar métricas sin salir del notebook (aunque en la UI de MLflow esta comparación es visual e interactiva).

In [ ]:
runs_df = mlflow.search_runs(order_by=["metrics.f1_score DESC"])
columnas_interes = [c for c in runs_df.columns if c.startswith("metrics.") or c.startswith("params.") or c in ["run_id", "tags.mlflow.runName"]]
runs_df[columnas_interes].head(10)


In [ ]:
mejor_run = runs_df.iloc[0]
print("Mejor run:", mejor_run["tags.mlflow.runName"])
print("run_id:", mejor_run["run_id"])
print("F1-score:", mejor_run["metrics.f1_score"])
print("Accuracy:", mejor_run["metrics.accuracy"])


## 6. Registrando el modelo ganador en el Model Registry

Una vez identificado el mejor *run*, lo registramos como una **versión de modelo** en el Model Registry. Esto crea (o reutiliza) un modelo con nombre, y le asigna un número de versión incremental.

In [ ]:
MODEL_NAME = "clase3_wine_classifier"
model_uri = f"runs:/{mejor_run['run_id']}/model"

try:
    registered_model = mlflow.register_model(model_uri=model_uri, name=MODEL_NAME)
    print(f"Modelo registrado: {registered_model.name} — versión {registered_model.version}")
except Exception as e:
    print("El registro requiere un Model Registry activo (disponible en Databricks Workspace).")
    print("Error:", e)


## 7. Promoviendo el modelo a `Staging`

El Model Registry maneja *stages*: `None → Staging → Production → Archived`. Promover una versión es responsabilidad del equipo (o de un pipeline de CI/CD) una vez que el modelo pasó sus validaciones.

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

try:
    client.transition_model_version_stage(
        name=MODEL_NAME,
        version=registered_model.version,
        stage="Staging",
        archive_existing_versions=False,
    )
    print(f"Versión {registered_model.version} de '{MODEL_NAME}' promovida a Staging ✅")
except NameError:
    print("Ejecuta primero la celda de registro del modelo (sección 6).")
except Exception as e:
    print("La transición de stage requiere un Model Registry activo en Databricks.")
    print("Error:", e)


## 8. Cargando el modelo desde el Registry (como lo haría producción)

Así es como una API o un pipeline de inferencia cargaría el modelo en producción: **sin conocer el código de entrenamiento**, solo referenciando su nombre y stage.

In [ ]:
try:
    modelo_produccion = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}/Staging")
    ejemplo = X_test.iloc[:5]
    predicciones = modelo_produccion.predict(ejemplo)
    print("Predicciones del modelo cargado desde el Registry:", predicciones)
    print("Valores reales:                                    ", list(y_test.iloc[:5]))
except Exception as e:
    print("Carga desde el Registry requiere que el modelo haya sido registrado y promovido (secciones 6 y 7).")
    print("Alternativa: cargar directamente desde el run_id.")
    modelo_desde_run = mlflow.sklearn.load_model(f"runs:/{mejor_run['run_id']}/model")
    print("Modelo cargado directamente desde el run:", modelo_desde_run)


## 🧪 Ejercicio propuesto

1. Entrena una **cuarta variante** del modelo cambiando `n_estimators` y `max_depth` a valores distintos.
2. Registra el nuevo *run* y compáralo contra los tres anteriores usando `mlflow.search_runs()`.
3. Si el nuevo modelo supera al actual en `f1_score`, regístralo como una **nueva versión** del mismo modelo (`clase3_wine_classifier`) y promuévela a `Staging`, archivando la versión anterior (`archive_existing_versions=True`).
4. Abre la pestaña **Experiments** en el Workspace de Databricks y compara los runs visualmente (gráficos de paralelo-coordenadas, scatter de métricas).

---
### Cierre del laboratorio
Completaste el ciclo de vida de un modelo con MLflow: entrenar → trackear → comparar → registrar → promover → servir. Este es el mismo flujo que usa un equipo de MLOps en producción.

**Próxima clase →** Laboratorio Práctico Dirigido: Proyecto Final Independiente, integrando Databricks + un modelo de ML/DL o NLP + MLflow de punta a punta.
